In [ ]:
import configparser
import os
import sys
from  datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Setup print length for pandas dataframes
pd.set_option('display.max_colwidth', None)
pd.set_option('expand_frame_repr', False)


In [ ]:
curr_dir = str(Path.cwd())
sys.path.append(str(Path.cwd().parent))

from etrade_client.auth.auth import EtradeAuthorization
from etrade_client.accounts.accounts import Accounts
from etrade_client.market.market import Market
from etrade_client.option_chain.option_chain import OptionChain
from etrade_client.research.research import OptionsResearchDelta

In [ ]:
# Read config file
config = configparser.ConfigParser()
ini_file = Path(curr_dir) / "config.ini"
config.read(ini_file)
cfg = config["DEFAULT"]
base_url = cfg["PROD_BASE_URL"] #SANDBOX_BASE_URL

In [ ]:
auth = EtradeAuthorization()
auth_result = auth.authorize(cfg, base_url)
session, base_url = auth_result.session, auth_result.base_url
market = Market(session, base_url)

In [ ]:
colSet = ['optionType', 'delta', 'strikePrice', 'bid', 'ask', 'lastPrice', 'volume', 'openInterest']

def get_options_research(chain_df, opt_type, dmin=0.05, dmax=0.3, query_param=None):
    option_mask = (chain_df['optionType'] == opt_type)
    delta_mask = (abs(chain_df['delta']) >= dmin) & ((abs(chain_df['delta']) < dmax))
    chain_df = chain_df[option_mask & delta_mask]
    if query_param:
        chain_df = chain_df.query(query_param)
    chain_df = chain_df[colSet]
    chain_df = chain_df.assign(mark=lambda x: (x.ask + x.bid) / 2)
    return chain_df

def extend_options_research(chain_df, cost_basis, d, qty=100):
    chain_df = chain_df.assign(gain_total=lambda x: x.mark*qty)
    chain_df = chain_df.assign(annual_rate=lambda x: 100*x.mark/cost_basis*365/d)
    return chain_df

In [ ]:
ticker = "GOOG"
qty = 100
# cost_basis = 10
cost_basis = market.quote(ticker)['lastPrice'][0]
opt_type = "CALL"

chain = OptionChain(session, base_url, ticker, as_dataframe=True)
chain.getExpiryDates()[:4]

In [ ]:
dates = ["06-18-2026", "07-17-2026"]
# dates = ["05-08-2026", "05-15-2026", "05-29-2026"]
# dates = ["05-15-2026", "05-29-2026"]
# dates = ["05-15-2026"]
exp_chain = {(datetime.strptime(d, "%m-%d-%Y") - datetime.today()).days: chain.view(d) for d in dates}

In [ ]:
print(cost_basis)

list(exp_chain.keys())

In [ ]:
print("Ticker:", ticker)
for d, chain_df in exp_chain.items():
    res1 = get_options_research(chain_df, opt_type)
    chain_research = extend_options_research(res1, cost_basis, d, qty)
    print(f"Days to Expiry: {d}")
    print(chain_research)


In [ ]:
# Sector: OIL
tickers = ["OXY", "CVX", "XOM", "PSX"]
cost_bases = [market.quote(t)['lastPrice'][0] for t in tickers]
opt_type = "CALL"
qty = 100

chains = {t: OptionChain(session, base_url, t, as_dataframe=True) for t in tickers}


In [ ]:
chain_research_df = []
for ticker, chain in chains.items():
    cost_basis = cost_bases[tickers.index(ticker)]
    print(f"\n    !!!!    Ticker: {ticker}  - Cost Basis: {cost_basis}    !!!!\n")
    # dates = ["06-18-2026", "07-17-2026"]
    dates = chain.getExpiryDates()["expiryDate"]
    # print("All Expiry Dates:", dates)
    # No longer than a month to expiry
    dates = [d for d in dates if 0 <= (datetime.strptime(d, "%m-%d-%Y") - datetime.today()).days <= 30]
    # dates = ["05-22-2026", "05-29-2026", "06-05-2026", "06-12-2026", "06-18-2026"]
    exp_chain = {(datetime.strptime(d, "%m-%d-%Y") - datetime.today()).days: chain.view(d) for d in dates}
    ret_chain_df = []
    for d, chain_df in exp_chain.items():
        res1 = get_options_research(chain_df, opt_type=opt_type, dmin=0.05, dmax=0.4)
        chain_research = extend_options_research(res1, cost_basis, d, qty)
        date = dates[list(exp_chain.keys()).index(d)]
        print(f"Days to Expiry: {d}, date: {date}")
        print(chain_research)
        ret_chain_df.append(chain_research)
    chain_research_df.append(ret_chain_df)